# Story 8: Recover from Failed Jobs
Your overnight workflow partially failed: one calculation raised a Python exception because of a bug, another job was
killed by the cluster - a node failure, the out-of-memory killer or the wall-time limit. You want to find out what
failed and why, fix it, and rerun *only* the failed calculations while reusing the successful ones.

**With plain SLURM** you identify the failed jobs by combing through `sacct` and log files, write a new batch script
which contains only the failed parameter sets, and merge the results of both rounds by hand.

**With executorlib** exceptions are returned through the futures and collected in an error log file, the status of
every call is recorded in the cache, and resubmitting the workflow only reruns the calls without a valid result - once
you remove their failed entries from the cache, as shown below.

This notebook runs on the **login node** of a SLURM cluster.

## Cluster-specific settings
**Adapt the following cell to your cluster.** It collects everything which depends on the cluster rather than on your science. The values below are the ones used to test this notebook on a single-node SLURM test cluster. The submission template is a [Jinja2](https://jinja.palletsprojects.com) template which executorlib renders for every function call: add the lines your cluster requires, for example `#SBATCH --partition=<name>`, `#SBATCH --account=<project>` or `module load ...` commands before `{{command}}`. The [executorlib-cluster-setup](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#setup-with-claude-code) skill for Claude Code can analyse your cluster and generate a tested template.

In [ ]:
# Cluster-specific settings - adapt these values to your cluster.
submission_template = """\
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --job-name={{job_name}}
#SBATCH --chdir={{working_directory}}
#SBATCH --get-user-env
#SBATCH --ntasks={{cores}}
{%- if run_time_max %}
#SBATCH --time={{ [1, run_time_max // 60]|max }}
{%- endif %}
{%- if dependency_list %}
#SBATCH --dependency=afterok:{{ dependency_list | join(',') }}
{%- endif %}

{{command}}
"""

## Step 1: A workflow with two kinds of failures
`compute_energy()` contains a bug - it divides by the strain, which fails for zero strain. In addition, the job for
one strain is killed on its first attempt, which simulates a transient node failure or the out-of-memory killer. Each
call gets an explicit `cache_key`, which names its cache files. Unlike the default hash, the `cache_key` does not
change when the code of the function changes - this is what allows fixing the function later while keeping the
successful results.

In [ ]:
import os
import signal
import time

marker_file = os.path.abspath("story5_node_failure.marker")


def simulate_node_failure(strain):
    # Kill the job on its first attempt for one strain - like a node failure or the out-of-memory killer would.
    if strain == 0.03 and not os.path.exists(marker_file):
        open(marker_file, "w").close()
        os.kill(os.getpid(), signal.SIGKILL)


def compute_energy(strain):
    # Stand-in for a simulation - with a bug: it fails for strain == 0.
    time.sleep(1)
    simulate_node_failure(strain)
    energy = 100.0 * strain**2
    return {"strain": strain, "energy": energy, "energy_per_strain": energy / strain}


strain_lst = [0.0, 0.01, 0.02, 0.03]

In [ ]:
from executorlib import SlurmClusterExecutor

cache_directory = os.path.abspath("story5_cache")
error_log_file = os.path.abspath("story5_errors.log")

with SlurmClusterExecutor(
    cache_directory=cache_directory,
    resource_dict={"submission_template": submission_template, "run_time_max": 300, "error_log_file": error_log_file},
) as exe:
    future_dict = {
        strain: exe.submit(compute_energy, strain, resource_dict={"cache_key": f"strain_{strain}"})
        for strain in strain_lst
    }
    for strain, future in future_dict.items():
        print(strain, repr(future.exception()))

Both failures surface through the future objects, so a workflow never waits forever:

* the Python exception (`ZeroDivisionError`) is raised by `result()` like for a local function,
* for the killed job no result was written. executorlib queries SLURM for the status of jobs without result -
  throttled to once every 30 seconds per job - and fails the future with a `RuntimeError` once SLURM reports the job as
  failed or no longer knows it. Detecting such a job therefore takes up to a minute.

## Step 2: Find out what failed
The `error_log_file` contains the traceback of every Python exception together with the function and its arguments:

In [ ]:
with open(error_log_file) as f:
    print(f.read())

The cache tells the full story. Each call is represented by one file, named after its `cache_key`:

* `<cache_key>_o.h5` with an `output`: the call finished successfully (`status` is `finished`),
* `<cache_key>_o.h5` with an `error`: the Python function raised an exception, the exception is stored in the cache,
* `<cache_key>_i.h5` without output: the job was submitted but never wrote a result. Once the job left the queue the
  `status` is `aborted`.

In [ ]:
import pandas
from executorlib import get_cache_data_queue

df = pandas.DataFrame(get_cache_data_queue(cache_directory=cache_directory, queue_type="slurm"))
df["file"] = df["filename"].apply(os.path.basename)
df[["file", "queue_id", "status", "error"]].sort_values("file")

## Step 3: Clear the failed entries
What happens when a call is resubmitted with the same `cache_directory` depends on its cache file:

| Cache file | Behaviour on resubmission |
|---|---|
| `_o.h5` with `output` | the result is loaded, no job is submitted |
| `_o.h5` with `error` | the **stored exception is raised again**, no job is submitted |
| `_i.h5` of a running job | the call is re-attached to the running job |
| `_i.h5` of a dead job | executorlib asks SLURM about the recorded job ID. Depending on the SLURM configuration the job is either resubmitted or - when the accounting database still reports the old job as failed - the old job is re-attached and the future fails again |

So executorlib does not retry failed calls on its own: a failed call is only rerun after its cache file is removed.
This is deliberate - a deterministic exception would just fail again, and an expensive calculation should not be
resubmitted without your decision. The following cell removes the entries of both kinds of failures. Calls
which depend on a failed call store the same exception, so they are removed as well. Unfinished jobs which are still
in the queue are not touched.

In [ ]:
for entry in get_cache_data_queue(cache_directory=cache_directory, queue_type="slurm"):
    python_error = "error" in entry
    job_died = "output" not in entry and entry["status"] == "aborted"
    if python_error or job_died:
        print("remove", os.path.basename(entry["filename"]))
        os.remove(entry["filename"])

To remove a single call, delete `<cache_directory>/<cache_key>_o.h5` (exception) or call
`terminate_task_in_cache(cache_directory=..., cache_key=..., backend="slurm")`, which cancels the job if it is still
in the queue and removes its `_i.h5` file.

## Step 4: Fix the bug and rerun
The fixed function keeps the same `cache_key`s. Only the two removed calls are submitted again, the successful
results are loaded from the cache. Since the `resource_dict` is not part of the cache key, a job which was killed by
the wall-time or memory limit can be resubmitted with a larger `run_time_max` or more memory in the same way.

In [ ]:
def compute_energy(strain):
    # Fixed version: handles strain == 0.
    time.sleep(1)
    simulate_node_failure(strain)
    energy = 100.0 * strain**2
    return {"strain": strain, "energy": energy, "energy_per_strain": energy / strain if strain != 0 else 0.0}


with SlurmClusterExecutor(
    cache_directory=cache_directory,
    resource_dict={"submission_template": submission_template, "run_time_max": 300, "error_log_file": error_log_file},
) as exe:
    future_dict = {
        strain: exe.submit(compute_energy, strain, resource_dict={"cache_key": f"strain_{strain}"})
        for strain in strain_lst
    }
    result_lst = [future.result() for future in future_dict.values()]

pandas.DataFrame(result_lst)

The SLURM job IDs show which results were computed in the second round:

In [ ]:
df = pandas.DataFrame(get_cache_data_queue(cache_directory=cache_directory, queue_type="slurm"))
df["file"] = df["filename"].apply(os.path.basename)
df[["file", "queue_id", "status"]].sort_values("queue_id")

**A note on dependencies:** when a job dies, SLURM keeps the jobs which depend on it pending with the reason
`DependencyNeverSatisfied`, unless the cluster is configured to cancel them. Cancel them with
`terminate_tasks_in_cache()` (see [Story 1](story-1-submit-and-reconnect.ipynb)), clear the failed entry and resubmit
the workflow.

## Clean up

In [ ]:
import shutil

shutil.rmtree(cache_directory, ignore_errors=True)
for file_name in [error_log_file, marker_file]:
    if os.path.exists(file_name):
        os.remove(file_name)

## What you learned
* Python exceptions and jobs killed by SLURM both surface as exceptions of the future objects.
* `error_log_file` collects the tracebacks with the function arguments; `get_cache_data_queue()` shows which calls
  finished, raised an exception or died.
* Failed calls are not retried automatically: remove their cache files, then resubmit - successful calls come from the
  cache.
* A `cache_key` keeps the identity of a call stable when the function code is fixed.

**Reference:** [HPC Cluster Executor](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html), [write log files](https://executorlib.readthedocs.io/en/latest/1-single-node.html#write-log-files),
[cache](https://executorlib.readthedocs.io/en/latest/1-single-node.html#cache) and the [resource dictionary](https://executorlib.readthedocs.io/en/latest/trouble_shooting.html#resource-dictionary).